In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install "drive/MyDrive/Master Data Science/TFM/GITHUB/modelado_multidimensional_engagement-0.1.0-py3-none-any.whl"

Mounted at /content/drive
Processing ./drive/MyDrive/Master Data Science/TFM/GITHUB/modelado_multidimensional_engagement-0.1.0-py3-none-any.whl


In [2]:
import pandas as pd
import src.config as config
import os
# Entrenamiento (Noviembre)
from src.modelling.transforms import EngagementScaler
from src.modelling.similarity import KendallTauSimilarity
from src.modelling.spectral import SpectralClusteringModel

In [3]:
config.WORKPATH = "drive/MyDrive/Master Data Science/TFM/GITHUB"

In [ ]:
cols = config.FEATURES_GENERIC
workpath = config.WORKPATH
n_clusters = config.CLUSTER_GENERIC


df_train = pd.read_parquet(os.path.join(workpath, "processed", "training_201911", "aggregated_generic.parquet"))
df_test = pd.read_parquet(os.path.join(workpath, "processed", "validation_201912", "aggregated_generic.parquet"))

# 1. Ranking y normalización
scaler = EngagementScaler(features=cols)
df_train['MergeUE'] = None  # placeholder
df_ranked = scaler.fit(df_train)  # Calcula means y stds
df_normalized = scaler.transform(df_ranked)
df_normalized['cluster'] = None  # placeholder

# 2. Similitud y afinidad
similarity = KendallTauSimilarity(method='kendall')
affinity_matrix = similarity.fit_transform(df_normalized, features=cols)

# 3. Clustering y guardar artefactos
spectral_model = SpectralClusteringModel(n_clusters=config.CLUSTER_GENERIC, n_init=50)
labels = spectral_model.fit(affinity_matrix, df_normalized, features=cols)

# Guardar estadísticas de entrenamiento
spectral_model.set_training_statistics(df_ranked, features=cols)

# Obtener resultados
df_clusters = spectral_model.get_clusters_dataframe(df_normalized)

# Guardar artefactos
spectral_model.save_model_artifacts(os.path.join(workpath, "models", "nov_generic_artifacts.np"))



Computing label assignment using kmeans
Initialization complete
Iteration 0, inertia 0.012952737914356994.
Iteration 1, inertia 0.01048972177070568.
Iteration 2, inertia 0.010248080009655937.
Iteration 3, inertia 0.010132917347124909.
Iteration 4, inertia 0.010125245949796595.
Iteration 5, inertia 0.010102693161335774.
Iteration 6, inertia 0.010095470700229545.
Converged at iteration 6: strict convergence.
Initialization complete
Iteration 0, inertia 0.017114963667704142.
Iteration 1, inertia 0.010072071520769674.
Iteration 2, inertia 0.009599797080144629.
Iteration 3, inertia 0.009569763853847764.
Converged at iteration 3: strict convergence.
Initialization complete
Iteration 0, inertia 0.012251822195611611.
Iteration 1, inertia 0.010765114285404404.
Iteration 2, inertia 0.010583071255907548.
Iteration 3, inertia 0.010533617346692553.
Converged at iteration 3: strict convergence.
Initialization complete
Iteration 0, inertia 0.013406353973332601.
Iteration 1, inertia 0.0099987251390189

In [5]:
# ============= PROYECCIÓN (Diciembre) =============

# 1. Cargar estadísticas de entrenamiento
scaler_dec = EngagementScaler(features=cols)
scaler_dec.means = spectral_model.train_rank_means
scaler_dec.stds = spectral_model.train_rank_stds

# 2. Aplicar transformación (sin reajustar)
df_ranked_dec = df_test.copy()
df_train['MergeUE'] = None  # placeholder
df_ranked_dec = scaler_dec.rank(df_ranked_dec)
df_normalized_dec = scaler_dec.transform(df_ranked_dec)

# 3. Proyectar en espacio de centroides
df_normalized_dec["cluster"] = spectral_model.predict(df_normalized_dec[cols].values)

# 2. Similitud y afinidad
similarity = KendallTauSimilarity(method='kendall')
affinity_matrix_dec = similarity.fit_transform(df_normalized_dec, features=cols)

In [8]:
affinity_matrix_dec.to_parquet(os.path.join(workpath, "processed", "validation_201912", "affinity_matrix_generic.parquet"))
affinity_matrix.to_parquet(os.path.join(workpath, "processed", "training_201911", "affinity_matrix_generic.parquet"))